In [1]:
import json
from pathlib import Path
import numpy as np

result_file = Path(f"button_usage.jsonl")
if result_file.exists():
    data = list(map(lambda x: json.loads(x), open(result_file).readlines()))

    print(f"The number of collections: {len(data)}")

    learn_costs = np.array([x["learn_costs"] for x in data])
    review_costs = np.array([x["review_costs"] for x in data])

    learn_costs_mean = np.median(learn_costs, axis=0).round(3).tolist()
    review_costs_mean = np.median(review_costs, axis=0).round(3).tolist()
    print(f"Learn costs median: {learn_costs_mean}")
    print(f"Review costs median: {review_costs_mean}")

    first_rating_probs = np.array([x["first_rating_prob"] for x in data])
    review_rating_probs = np.array([x["review_rating_prob"] for x in data])

    first_rating_probs_mean = np.mean(first_rating_probs, axis=0).round(3).tolist()
    review_rating_probs_mean = np.nanmean(review_rating_probs, axis=0).round(3).tolist()
    print(f"First rating prob mean: {first_rating_probs_mean}")
    print(f"Review rating prob mean: {review_rating_probs_mean}")

    first_rating_offsets = np.array([x["first_rating_offset"] for x in data])
    first_session_lens = np.array([x["first_session_len"] for x in data])

    first_rating_offsets_median = (
        np.median(first_rating_offsets, axis=0).round(3).tolist()
    )
    first_session_lens_median = np.median(first_session_lens, axis=0).round(3).tolist()
    print(f"First rating offset median: {first_rating_offsets_median}")
    print(f"First session len median: {first_session_lens_median}")

    forget_rating_offsets = np.array([x["forget_rating_offset"] for x in data])
    forget_session_lens = np.array([x["forget_session_len"] for x in data])

    forget_rating_offsets_median = np.median(forget_rating_offsets).round(3)
    forget_session_lens_median = np.median(forget_session_lens).round(3)
    print(f"Forget rating offset median: {forget_rating_offsets_median}")
    print(f"Forget session len median: {forget_session_lens_median}")

    short_term_recall = np.array([x["short_term_recall"] for x in data])
    short_term_recall = short_term_recall[~(short_term_recall == 0).any(axis=1) & ~(short_term_recall == 1).any(axis=1)]
    short_term_recall_mean = np.mean(short_term_recall, axis=0).round(3).tolist()
    print(f"Short term recall mean: {short_term_recall_mean}")

The number of collections: 10000
Learn costs median: [34.545, 27.0, 14.16, 6.78]
Review costs median: [23.185, 11.81, 7.48, 5.64]
First rating prob mean: [0.236, 0.103, 0.489, 0.173]
Review rating prob mean: [0.231, 0.626, 0.143]
First rating offset median: [-0.75, -0.19, -0.01, 0.0]
First session len median: [2.04, 1.43, 0.82, 0.0]
Forget rating offset median: -0.27
Forget session len median: 1.06
Short term recall mean: [0.742, 0.917, 0.962, 0.853]


In [17]:
import pandas as pd
from typing_extensions import Tuple

# The percentage of use at which the rating is concidered "used"
USED_THRESHOLD = 0.015

def label(values: Tuple[bool, bool, bool, bool]):
    return ",".join(s for a,s in zip(values, ["again", "hard", "good", "easy"]) if a)

if result_file.exists():
    data = pd.DataFrame(list(map(lambda x: json.loads(x), open(result_file).readlines())), columns=["first_rating_prob", "review_rating_prob"])
    data["first_rating_usage"] = data["first_rating_prob"].apply(lambda arr: label(a > USED_THRESHOLD for a in arr))
    data["review_rating_usage"] = data["review_rating_prob"].apply(lambda arr: label((True, *(a > USED_THRESHOLD for a in arr))))
    data["user_count"] = 1 # Just for the column title

## Rating usage with reviews
(again use is assumed)

In [18]:
data.groupby("review_rating_usage")["user_count"].count().reset_index()

,review_rating_usage,user_count
0,again,12
1,"again,easy",52
2,"again,good",853
3,"again,good,easy",421
4,"again,hard",37
5,"again,hard,easy",21
6,"again,hard,good",2661
7,"again,hard,good,easy",5943


## Rating usage with initial reviews

In [19]:
data.groupby("first_rating_usage")["user_count"].count().reset_index()

,first_rating_usage,user_count
0,"again,easy",57
1,"again,good",836
2,"again,good,easy",1929
3,"again,hard",19
4,"again,hard,easy",7
5,"again,hard,good",1056
6,"again,hard,good,easy",4949
7,easy,7
8,good,139
9,"good,easy",217
